# Unit 08 - CUPED (Exercise) · **V2 material**

**Atoms:** `U08-A6` · **Runtime:** ~25 seconds

**After this notebook you can:** implement `CUPED` adjustment yourself and check when it pays off.

## Without code

Plain SE is about **0.19**; `CUPED` SE drops to roughly **0.12**, a variance reduction near **58%**. When the pre-period barely predicts the outcome, the same adjustment buys under **1%** - `CUPED` is only as good as the covariate.

## 1. The question

Implement plain and `CUPED` estimates on simulated revenue data and quantify the variance reduction.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Generate correlated pre-period and in-period revenue with a known `ATE`.

In [ ]:
n = 6000
true_ate = 1.5
pre = np.random.normal(40, 8, n)
y_base = 0.7 * pre + np.random.normal(0, 4, n)
treatment = np.random.binomial(1, 0.5, n)
y = y_base + true_ate * treatment + np.random.normal(0, 2.5, n)
df = pd.DataFrame({'pre': pre, 'y': y, 'treatment': treatment})

## 4. TODO - plain estimate

Fit `y ~ treatment`. Store `plain_se` = standard error of the treatment coefficient.

In [ ]:
plain_se = None  # TODO
assert plain_se is not None
print('Plain SE:', round(plain_se, 4))
assert plain_se > 0.01

## 5. TODO - CUPED estimate

Fit `y ~ treatment + pre`. Store `cuped_se` and `var_reduction = 1 - (cuped_se/plain_se)**2`.

In [ ]:
cuped_se = None  # TODO
var_reduction = None  # TODO
assert cuped_se is not None and var_reduction is not None
print('CUPED SE:', round(cuped_se, 4))
print('Variance reduction:', round(var_reduction * 100, 1), '%')
assert var_reduction > 0.25

## 6. TODO - weak correlation

Build a dataset where `pre` barely predicts `y` (multiply `pre` by 0.05 in the generating equation). `weak_vr` should be below 0.10.

In [ ]:
weak_vr = None  # TODO
assert weak_vr is not None
print('Weak-link variance reduction:', round(weak_vr * 100, 1), '%')
assert weak_vr < 0.10

**Takeaway:** `CUPED` needs a predictive pre-period signal. **Unit:** [V2 unit 08](../V2/units/unit-08-power-duration-sample-size/README.md)

## Hints

Use `smf.ols(...).fit().bse['treatment']` for standard errors.

## Spoiler

```python
plain_se = smf.ols('y ~ treatment', data=df).fit().bse['treatment']
cuped_se = smf.ols('y ~ treatment + pre', data=df).fit().bse['treatment']
var_reduction = 1 - (cuped_se / plain_se) ** 2

# Weak link: the pre-period barely predicts y, so adjusting for it buys almost nothing
pre_w = np.random.normal(40, 8, n)
y_w = (0.05 * pre_w + np.random.normal(0, 4, n)
       + true_ate * treatment + np.random.normal(0, 2.5, n))
df_w = pd.DataFrame({'pre': pre_w, 'y': y_w, 'treatment': treatment})
plain_se_w = smf.ols('y ~ treatment', data=df_w).fit().bse['treatment']
cuped_se_w = smf.ols('y ~ treatment + pre', data=df_w).fit().bse['treatment']
weak_vr = 1 - (cuped_se_w / plain_se_w) ** 2
```